# 00 · Orquestador del pipeline

Ejecuta los notebooks del pipeline **en orden** con `dbutils.notebook.run()`.
Si un paso falla (excepción o `status = FAIL`), se detienen los siguientes.

La consigna no permite Databricks Jobs, por eso la orquestación se simula con este notebook.
En producción se reemplazaría por un Job con las mismas tareas y dependencias (DT-010).

**Prerrequisito:** ejecutar los scripts de `setup/` (ver `setup/README.md`).

In [ ]:
dbutils.widgets.text("catalog", "workspace", "Catálogo")
CATALOG = dbutils.widgets.get("catalog")

## Pasos del pipeline
Para agregar un paso, sumarlo a `STEPS` en el orden en que debe ejecutarse.

In [ ]:
STEPS = [
    {"notebook": "./01_bronze_ingesta", "params": {"catalog": CATALOG, "source": "transacciones"}},
]

TIMEOUT_SECONDS = 3600

In [ ]:
import json
import time

results = []

for step in STEPS:
    started = time.time()
    try:
        output = json.loads(dbutils.notebook.run(step["notebook"], TIMEOUT_SECONDS, step["params"]))
        status = output.get("status", "FAIL")
        message = output.get("message", "")
    except Exception as e:
        status, message = "FAIL", f"{type(e).__name__}: {e}"

    results.append({
        "paso": step["notebook"],
        "status": status,
        "duracion_seg": round(time.time() - started, 1),
        "detalle": message,
    })

    if status == "FAIL":
        break

## Resumen de la corrida

In [ ]:
skipped = [s["notebook"] for s in STEPS[len(results):]]
for notebook in skipped:
    results.append({"paso": notebook, "status": "SKIPPED", "duracion_seg": 0.0, "detalle": "No ejecutado por falla previa"})

display(spark.createDataFrame(results, "paso STRING, status STRING, duracion_seg DOUBLE, detalle STRING"))

failed = [r for r in results if r["status"] == "FAIL"]
assert not failed, f"Pipeline con fallas: {failed}"
print("Pipeline completado")